# 5. 日々の株価分析

### ライブラリのインポート

分析に必要なライブラリを読み込みます。

| ライブラリ | 用途 |
|-----------|------|
| `os`, `sys` | 環境変数・モジュールパスの操作 |
| `dotenv` | `.env` ファイルからの環境変数読み込み |
| `requests` | バックエンドAPIへのHTTPリクエスト |
| `plotly.graph_objs` | インタラクティブなグラフの描画 |
| `talib` | テクニカル指標の計算（TA-Lib） |
| `datetime` | 日付・時刻の操作 |
| `pandas` | 時系列データの整形・集計 |
| `numpy` | 数値計算・配列操作 |
| `mplfinance` | ローソク足チャートの描画 |
| `matplotlib` | 静的グラフの描画 |
| `backtesting` | バックテストフレームワーク |
| `Modules.rci` | RCI（順位相関係数）計算カスタムモジュール |

In [1]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path
import requests

import plotly.graph_objs as go
import talib as ta
import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mplfinance.original_flavor import candlestick_ohlc
import mplfinance as mpf
import matplotlib.dates as mdates
from backtesting import Backtest, Strategy
from Modules.rci import Rci

/workspace/.venv/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

### API接続設定・モジュール読み込み

- **`API_BASE_URL`**: バックエンドAPIのベースURL。  
  環境変数 `API_BASE_URL` から取得し、未設定の場合は `http://localhost:8000` をデフォルト値として使用します。  
  Docker Compose 環境では通常 `http://api:8000` が設定されます。
- **`GetMarketData`**: yfinance を利用した株価データ取得モジュール。  
  `/workspace/data` をデータキャッシュパスに設定します。

In [2]:
# APIの接続先（notebookコンテナでは通常 http://api:8000 が設定される）
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
# データを取得する関数
sys.path.append("/workspace/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/workspace/data'))

## 5.2 複数銘柄の比較検討

#### トヨタ自動車（7203.T）データの登録

`POST /api/v1/stock/data` エンドポイントに対して、銘柄コード・市場・期間を指定してリクエストを送信します。  
バックエンドは yfinance から株価データを取得し、`mst_stock`（銘柄マスタ）と `trn_stock_price`（株価履歴）テーブルへ UPSERT します。

| パラメータ | 値 | 説明 |
|-----------|----|------|
| `code` | `7203` | 東証銘柄コード |
| `market` | `TSE` | 東京証券取引所（Tokyo Stock Exchange） |
| `name` | トヨタ自動車 | 銘柄名 |
| `start` / `end` | 直近2年 | 取得期間 |

In [3]:
# 直近2年の期間を作成
end_date = dt.datetime(2024, 12, 31)
start_date = end_date - dt.timedelta(days=365 * 2)

# トヨタ自動車 (Yahoo Finance: 7203.T)
code = "7203"
market = "TSE"

post_payload = {
    "code": code,
    "market": market,
    "name": "トヨタ自動車",
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

post_url = f"{API_BASE_URL}/api/v1/stock/data"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())

POST URL: http://api:8000/api/v1/stock/data
POST status: 200
POST response: {'result': True}


#### トヨタ自動車（7203.T）データの取得

`GET /api/v1/stock/get` エンドポイントで登録済みの株価データを照会します。  
取得した JSON レスポンスの `results` キーに含まれる `list[dict]` を `pd.DataFrame` に変換し、`df_results_toyota` として保存します。

In [4]:
get_url = f"{API_BASE_URL}/api/v1/stock/get"
get_params = {
    "code": code,
    "market": market,
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_results_toyota = pd.DataFrame(results)

    print("取得件数:", len(df_results_toyota))
    display(df_results_toyota.head())
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/stock/get
GET status: 200
取得件数: 491


,code,market,date,open,high,low,close,volume
0,7203,TSE,2023-01-04,1620.543579,1625.047591,1610.184351,1619.642777,25995600
1,7203,TSE,2023-01-05,1628.200439,1639.010069,1615.589205,1632.254051,24700200
2,7203,TSE,2023-01-06,1643.964478,1647.567687,1626.849231,1630.002040,22568600
3,7203,TSE,2023-01-10,1645.765991,1666.484446,1640.811578,1655.224416,22352300
4,7203,TSE,2023-01-11,1655.224487,1657.476493,1641.262049,1643.063654,19798400


#### ソニーグループ（6758.T）データの登録

トヨタ自動車と同様に、ソニーグループの株価データを `POST /api/v1/stock/data` で登録します。

| パラメータ | 値 | 説明 |
|-----------|----|------|
| `code` | `6758` | 東証銘柄コード |
| `market` | `TSE` | 東京証券取引所（Tokyo Stock Exchange） |
| `name` | ソニーグループ | 銘柄名 |

In [5]:
# 直近2年の期間を作成
end_date = dt.datetime(2024, 12, 31)
start_date = end_date - dt.timedelta(days=365 * 2)

# ソニーグループ (Yahoo Finance: 6758.T)
code = "6758"
market = "TSE"

post_payload = {
    "code": code,
    "market": market,
    "name": "ソニーグループ",
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

post_url = f"{API_BASE_URL}/api/v1/stock/data"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())

POST URL: http://api:8000/api/v1/stock/data
POST status: 200
POST response: {'result': True}


#### ソニーグループ（6758.T）データの取得

`GET /api/v1/stock/get` でソニーグループの株価データを取得し、`df_results_sony` として保存します。  
以降の比較分析では `df_results_toyota` と `df_results_sony` を統合して使用します。

In [6]:
get_url = f"{API_BASE_URL}/api/v1/stock/get"
get_params = {
    "code": code,
    "market": market,
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_results_sony = pd.DataFrame(results)

    print("取得件数:", len(df_results_sony))
    display(df_results_sony.head())
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/stock/get
GET status: 200
取得件数: 491


,code,market,date,open,high,low,close,volume
0,6758,TSE,2023-01-04,1986.574585,2014.912343,1961.168319,1977.780108,20124500
1,6758,TSE,2023-01-05,2029.569458,2053.998556,2012.957671,2014.911999,21693000
2,6758,TSE,2023-01-06,2078.427734,2089.176538,2023.706553,2027.615209,20535000
3,6758,TSE,2023-01-10,2096.016602,2134.125994,2078.427651,2123.377191,19630000
4,6758,TSE,2023-01-11,2169.303955,2174.189775,2110.674118,2120.445758,22694500


### 5.2.1 複数銘柄の株価データをまとめる

複数銘柄のデータをループで一括取得し、`(Stock, date)` をキーとする **MultiIndex DataFrame** に統合します。

処理の流れ：
1. `company_names` リストに含まれる銘柄を順番に `GET /api/v1/stock/get` へリクエスト
2. 取得した `list[dict]` を `pd.DataFrame` に変換
3. `date` 列を `datetime.date` 型に統一
4. 銘柄コードを示す `Stock` 列を追加
5. `pd.concat` で全銘柄を縦結合し、`set_index(["Stock", "date"])` で MultiIndex 化
6. `filter_start`〜`filter_end` の範囲で `filtered_df` を抽出

In [7]:
get_url = f"{API_BASE_URL}/api/v1/stock/get"

# 直近2年の期間を作成
end_date = dt.datetime(2024, 12, 31)
start_date = end_date - dt.timedelta(days=365 * 2)

# 空のリストを用意
data_frames = []

# 銘柄リスト
company_names = [
    {"code": "7203", "market": "TSE", "name": "トヨタ自動車"},
    {"code": "6758", "market": "TSE", "name": "ソニーグループ"},
]

for company in company_names:
    code = company["code"]
    market = company["market"]

    # APIへRequestしデータを取得
    get_params = {
        "code": code,
        "market": market,
        "start": start_date.date().isoformat(),
        "end": end_date.date().isoformat(),
    }

    response = requests.get(get_url, params=get_params, timeout=60)
    if response.status_code != 200:
        print(f"GET failed: code={code}, status={response.status_code}, body={response.text}")
        continue

    payload = response.json()
    rows = payload.get("results", [])
    if not rows:
        print(f"No rows returned: code={code}")
        continue

    # APIレスポンス(list[dict])からDataFrameを作成
    df = pd.DataFrame(rows)

    # date列を日付型に揃える
    df["date"] = pd.to_datetime(df["date"]).dt.date

    # 銘柄コードを追加
    df["Stock"] = str(code)

    # 統合用のリストに追加
    data_frames.append(df)

if not data_frames:
    raise ValueError("対象期間のデータが1件も取得できませんでした")

# 全データを連結
df = pd.concat(data_frames, ignore_index=True)

# インデックスを設定（マルチインデックス化）
df = df.set_index(["Stock", "date"])

# 指定した期間のデータを抽出
filter_start = dt.date(2024, 1, 1)
filter_end = dt.date(2024, 12, 31)

filtered_df = df.loc[
    (df.index.get_level_values("date") >= filter_start)
    & (df.index.get_level_values("date") <= filter_end)
]

filtered_df.head()

code market         open         high          low  \
Stock date                                                             
7203  2024-01-04  7203    TSE  2446.039551  2446.039551  2387.557391   
      2024-01-05  7203    TSE  2507.770752  2513.804626  2453.930032   
      2024-01-09  7203    TSE  2500.808594  2521.230936  2471.567513   
      2024-01-10  7203    TSE  2548.151367  2554.649385  2487.348484   
      2024-01-11  7203    TSE  2640.051758  2661.866531  2624.270858   

                        close    volume  
Stock date                               
7203  2024-01-04  2418.190903  29812900  
      2024-01-05  2453.930032  30515500  
      2024-01-09  2520.302647  30486100  
      2024-01-10  2491.525781  33701200  
      2024-01-11  2624.270858  49003100

### 5.2.2 累積リターンの比較

#### 日次リターン・累積資産額の計算

**日次リターン**（単純収益率）は前日終値に対する変化率で定義します：

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**累積リターン**を考慮した資産額は、初期投資額 $I_0$ と各日のリターンの積累積から算出します：

$$V_t = I_0 \cdot \prod_{i=1}^{t}(1 + r_i)$$

pandas では `pct_change()` で $r_t$ を、`cumprod()` で累積積を計算します。  
初期投資額 $I_0 = 100{,}000$ 円（10万円）として銘柄間のパフォーマンスを比較します。

In [8]:
# 結果を格納する空のリスト
result_list = []

# 初期投資額
initial_investment = 100000  # 10万円

# filtered_dfから銘柄コードを取得
codes = filtered_df.index.get_level_values("Stock").unique()

for ticker in codes:
    # 銘柄ごとにデータを抽出
    ticker_data = filtered_df.loc[ticker].copy()  # コピーを作成
    ticker_data = ticker_data.reset_index()
    
    # 日次リターンを計算
    ticker_data["Daily Return"] = ticker_data["close"].pct_change()
   
    # 累積リターンを考慮した資産額を算出
    ticker_data["Asset Value"] = initial_investment * (1 + ticker_data["Daily Return"]).cumprod()

    # データフレームをリストに格納
    ticker_data["Stock"] = ticker
    result_list.append(ticker_data)

# 全データを連結
df = pd.concat(result_list)
df = df.set_index(["Stock", "date"]) 


#### 計算結果の確認

各銘柄の直近5件について `Daily Return`（日次リターン）と `Asset Value`（累積資産額）を表示します。  
最終的な資産額が初期投資額 $I_0 = 100{,}000$ 円に対してどれだけ変動したかを確認できます。

In [9]:
df[["Daily Return", "Asset Value"]].groupby(level="Stock").tail(5)

Daily Return    Asset Value
Stock date                                   
7203  2024-12-24      0.016043  112346.071967
      2024-12-25     -0.007018  111557.672887
      2024-12-26      0.060071  118259.019689
      2024-12-27      0.068000  126300.640901
      2024-12-30     -0.006866  125433.400906
6758  2024-12-24     -0.006272  125661.652854
      2024-12-25     -0.010821  124301.912328
      2024-12-26      0.002735  124641.848294
      2024-12-27      0.030303  128418.876964
      2024-12-30      0.008824  129551.994029

#### 累積資産額の時系列チャート（Plotly）

`plotly.graph_objs.Scatter` を使い、各銘柄の `Asset Value` を一枚のグラフに重ね書きします。  
x 軸には `pd.date_range(freq="ME")` で生成した月末日付を目盛りとして表示し、  
ホバーテキストには銘柄名・日付・資産額（円）を整形して表示します。

In [10]:
# ラベルを等間隔で設定し毎月末の日付を表示
tick_vals = pd.date_range(
    start=df.index.get_level_values("date").min(),
    end=df.index.get_level_values("date").max(),
    freq="ME"  # 月末
)
tick_vals = [pd.Timestamp(df.index.get_level_values("date").min())] + list(tick_vals) + [pd.Timestamp(df.index.get_level_values("date").max())]
tick_vals = sorted(set(tick_vals))  # 重複を排除し昇順に並べる

# Plotlyで可視化
fig = go.Figure()

for i, ticker in enumerate(codes):
    # 各銘柄のデータを抽出
    ticker_data = df.loc[ticker]
    # 銘柄コードから企業名を取得（なければコードを使用）
    name = company_names[i]['name']

    # 資産額のプロット
    fig.add_trace(go.Scatter(
        x=ticker_data.index.get_level_values("date"),
        y=ticker_data["Asset Value"],
        mode="lines",
        name=name,
        hoverinfo="text",
        text=[
            f"<b>{date}</b><br>{name} 資産額: {value:,.0f} 円"
            for date, value in zip(ticker_data.index.get_level_values("date"), ticker_data["Asset Value"])
        ]
    ))

# グラフの設定
fig.update_layout(
    title="資産額の比較",
    xaxis=dict(
        title="日付",
        showline=True,
        showgrid=True,
        tickformat="%Y-%m-%d",  # 日付フォーマット
        tickvals=tick_vals,  # ラベルを表示する日付
        ticktext=[str(t.date()) for t in tick_vals],  # ラベルとして表示するテキスト
    ),
    yaxis=dict(
        title="資産額（円）",
        showline=True
    ),
    template="plotly_white",
)
# グラフを表示
fig.show()


### 5.2.3 ボラティリティの比較

#### ボラティリティとは

**ボラティリティ（Volatility）** は株価変動の大きさを定量化したリスク指標です。  
一般に日次リターンの標準偏差で表現します。

---

**日次リターン** $r_t$ を次のように定義します：

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

**移動ボラティリティ（Rolling Volatility）** は、直近 $N$ 日間の日次リターンの標本標準偏差です：

$$\sigma_t^{(N)} = \sqrt{\frac{1}{N-1}\sum_{i=0}^{N-1}\left(r_{t-i} - \bar{r}_t^{(N)}\right)^2}$$

ここで $\bar{r}_t^{(N)} = \dfrac{1}{N}\displaystyle\sum_{i=0}^{N-1}r_{t-i}$ は直近 $N$ 日の平均リターンです。

このノートブックでは $N=20$（約1ヶ月分の取引日数）を使用します。  
最初の20行は計算に必要なデータが不足するため `NaN` になります。

---

**年率換算ボラティリティ** は日次ボラティリティに年間取引日数の平方根を掛けて求めます：

$$\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$$

| ボラティリティの解釈 | |
|----------------------|--|
| 高い（例: 0.03以上/日） | 価格変動が大きく、リスクが高い |
| 低い（例: 0.01以下/日） | 価格が安定しており、リスクが低い |

---

まずボラティリティ計算に使用するデータを再取得します。

In [11]:
# 空のリストを用意
data_frames = []

for company in company_names:
    code = company["code"]
    market = company["market"]

    # APIへRequestしデータを取得
    get_params = {
        "code": code,
        "market": market,
        "start": start_date.date().isoformat(),
        "end": end_date.date().isoformat(),
    }

    response = requests.get(get_url, params=get_params, timeout=60)
    if response.status_code != 200:
        print(f"GET failed: code={code}, status={response.status_code}, body={response.text}")
        continue

    payload = response.json()
    rows = payload.get("results", [])
    if not rows:
        print(f"No rows returned: code={code}")
        continue

    # APIレスポンス(list[dict])からDataFrameを作成
    df = pd.DataFrame(rows)

    # date列を日付型に揃える
    df["date"] = pd.to_datetime(df["date"]).dt.date

    # 銘柄コードを追加
    df["Stock"] = str(code)

    # 統合用のリストに追加
    data_frames.append(df)

# 全データを連結
df = pd.concat(data_frames)

# インデックスを設定（マルチインデックス化）
df = df.set_index(["Stock", "date"])  

# 指定した期間のデータを抽出
start_date = dt.date(2024, 1, 1)
end_date = dt.date(2024, 12, 31)

filtered_df = df.loc[(df.index.get_level_values("date") >= start_date) & 
                           (df.index.get_level_values("date") <= end_date)]
filtered_df.head()


code market         open         high          low  \
Stock date                                                             
7203  2024-01-04  7203    TSE  2446.039551  2446.039551  2387.557391   
      2024-01-05  7203    TSE  2507.770752  2513.804626  2453.930032   
      2024-01-09  7203    TSE  2500.808594  2521.230936  2471.567513   
      2024-01-10  7203    TSE  2548.151367  2554.649385  2487.348484   
      2024-01-11  7203    TSE  2640.051758  2661.866531  2624.270858   

                        close    volume  
Stock date                               
7203  2024-01-04  2418.190903  29812900  
      2024-01-05  2453.930032  30515500  
      2024-01-09  2520.302647  30486100  
      2024-01-10  2491.525781  33701200  
      2024-01-11  2624.270858  49003100

#### 移動ボラティリティの算出

`pandas` の `rolling(window=20).std()` を用いて直近20日間の移動ボラティリティを算出します：

$$\sigma_t^{(20)} = \text{std}\bigl(r_{t-19},\, r_{t-18},\, \ldots,\, r_t\bigr)$$

- `pct_change()` で日次リターン $r_t$ を計算
- `rolling(window=20).std()` でウィンドウサイズ20の移動標準偏差を計算
- 先頭の20行は `NaN`（データ不足）になります

In [12]:
result_list = []

# filtered_dfから銘柄コードを取得
codes = filtered_df.index.get_level_values("Stock").unique()

for ticker in codes:
    # 銘柄ごとにデータを抽出
    ticker_data = filtered_df.loc[ticker].copy()  # コピーを作成
    ticker_data = ticker_data.reset_index()
    
    # 日次リターンの算出
    ticker_data['Daily Return'] = ticker_data['close'].pct_change()

    # ボラティリティ（20日間の移動標準偏差）の算出
    ticker_data['Volatility'] = ticker_data['Daily Return'].rolling(window=20).std()
    
    # データフレームをリストに格納
    ticker_data["Stock"] = ticker
    result_list.append(ticker_data)

# 全データを連結
df = pd.concat(result_list)
df = df.set_index(["Stock", "date"]) 

#### ボラティリティ計算結果の確認

各銘柄の直近5件のボラティリティ値を確認します。  
値が大きいほどその期間の価格変動が激しく、リスクが高い状態を表します。

In [13]:
df[["Volatility"]].groupby(level="Stock").tail(5)

Volatility
Stock date                  
7203  2024-12-24    0.015087
      2024-12-25    0.012470
      2024-12-26    0.017472
      2024-12-27    0.022102
      2024-12-30    0.021478
6758  2024-12-24    0.021220
      2024-12-25    0.021427
      2024-12-26    0.021367
      2024-12-27    0.022086
      2024-12-30    0.021898

#### ボラティリティの時系列チャート（Plotly）

各銘柄の `Volatility`（移動ボラティリティ）推移を `plotly.graph_objs.Scatter` で折れ線グラフに描画します。  
銘柄を重ね合わせることで、相対的なリスクの高低とその変化タイミングを視覚的に比較できます。

- ボラティリティの**急上昇**は市場の混乱や決算発表など大きなイベントを示唆します
- ボラティリティの**収束**は相場の落ち着きを意味します

In [14]:
# ラベルを等間隔で設定（例: 毎月末の日付を表示）
tick_vals = pd.date_range(
    start=df.index.get_level_values("date").min(),
    end=df.index.get_level_values("date").max(),
    freq="ME"  # 月末
)
tick_vals = [pd.Timestamp(df.index.get_level_values("date").min())] + list(tick_vals) + [pd.Timestamp(df.index.get_level_values("date").max())]
tick_vals = sorted(set(tick_vals))  # 重複を排除し昇順に並べる

# Plotlyを使用して可視化
fig = go.Figure()

for i, ticker in enumerate(codes):
    # 各銘柄のデータを抽出
    ticker_data = df.loc[ticker]
    # 銘柄コードから企業名を取得（なければコードを使用）
    name = company_names[i]['name']

    # グラフに追加
    fig.add_trace(go.Scatter(
        x=ticker_data.index,
        y=ticker_data['Volatility'],
        mode='lines',
        name=name
    ))

# グラフの設定
fig.update_layout(
    title="ボラティリティ比較",
    xaxis=dict(
        title="日付",
        showline=True,
        showgrid=True,
        tickformat="%Y-%m-%d",  # 日付フォーマット
        tickvals=tick_vals,  # ラベルを表示する日付
        ticktext=[str(t.date()) for t in tick_vals],  # ラベルとして表示するテキスト
    ),
    yaxis=dict(
        title="ボラティリティ",
        showline=True
    ),
    template="plotly_white",
)

# グラフを表示
fig.show()


### 5.2.4 シャープレシオの比較

#### シャープレシオとは

**シャープレシオ（Sharpe Ratio）** は、リスク1単位あたりの超過リターンを示す代表的なパフォーマンス評価指標です。  
ウィリアム・シャープ（William Sharpe）が1966年に提唱しました。

---

**定義式：**

$$S = \frac{E[R_p] - R_f}{\sigma_p}$$

| 記号 | 意味 |
|------|------|
| $E[R_p]$ | ポートフォリオ（銘柄）の期待リターン（日次平均） |
| $R_f$ | リスクフリー金利（日次換算） |
| $\sigma_p$ | リターンの標準偏差（日次） |
| $E[R_p] - R_f$ | 超過リターン（Excess Return） |

---

**日次換算のリスクフリー金利：**

年率リスクフリー金利 $r_f$ を252営業日で割り、日次換算します：

$$R_f^{\text{daily}} = \frac{r_f}{252}$$

このノートブックでは $r_f = 1\%$（日本国債利回りを参考）を仮定します：

$$R_f^{\text{daily}} = \frac{0.01}{252} \approx 0.0000397$$

---

**シャープレシオの解釈：**

| 値の範囲 | 評価 |
|----------|------|
| $S > 2$ | 非常に優れたリスク調整後リターン |
| $1 \leq S \leq 2$ | 良好。リスクに見合うリターンが得られている |
| $0 < S < 1$ | リターンはあるがリスクが相対的に大きい |
| $S \leq 0$ | リスクフリー資産を下回る（不良） |

値が高いほど「同じリスク水準に対してより多くの超過リターンを得ている」ことを意味し、  
銘柄・ファンド間の比較や戦略評価に広く用いられます。

---

まずシャープレシオ計算に使用するデータを準備します。

In [15]:
# 空のリストを用意
data_frames = []

for ticker in codes:
    # 銘柄ごとにデータを抽出
    ticker_data = filtered_df.loc[ticker].copy()  # コピーを作成
    ticker_data = ticker_data.reset_index()
    
    # 日次リターンの算出
    ticker_data['Daily Return'] = ticker_data['close'].pct_change()

    # ボラティリティ（20日間の移動標準偏差）の算出
    ticker_data['Volatility'] = ticker_data['Daily Return'].rolling(window=20).std()
    
    # データフレームをリストに格納
    ticker_data["Stock"] = ticker
    data_frames.append(ticker_data)

# 全データを連結
df = pd.concat(data_frames)

# インデックスを設定（マルチインデックス化）
df = df.set_index(["Stock", "date"])  

# 指定した期間のデータを抽出
start_date = dt.date(2024, 1, 1)
end_date = dt.date(2024, 12, 31)

filtered_df = df.loc[(df.index.get_level_values("date") >= start_date) & 
                           (df.index.get_level_values("date") <= end_date)]


#### シャープレシオの計算

各銘柄の期間全体の日次リターンから平均リターン $\bar{r}_p$ と標準偏差 $\sigma_p$ を求め、シャープレシオを算出します：

$$S = \frac{\bar{r}_p - R_f^{\text{daily}}}{\sigma_p}, \qquad \bar{r}_p = \frac{1}{T}\sum_{t=1}^{T}r_t$$

- `mean()` で期間平均の日次リターン $\bar{r}_p$ を計算
- `std()` でリターンの標準偏差 $\sigma_p$ を計算
- 算出したシャープレシオは全行に同一値として付与し、棒グラフ比較に使用します

In [16]:
result_list = []

# filtered_dfから銘柄コードを取得
codes = filtered_df.index.get_level_values("Stock").unique()

# リスクフリー金利
risk_free_rate = 0.01  # 1%と仮定
risk_free_rate_per_day = risk_free_rate / 252 # 252は年間の取引日数

for ticker in codes:
    # 銘柄ごとにデータを抽出
    ticker_data = filtered_df.loc[ticker].copy()  # 明確にコピーを作成
    ticker_data = ticker_data.reset_index()
    
    # 日次リターンを計算
    ticker_data['Daily Return'] = ticker_data['close'].pct_change()

    # 平均リターンとリスク（標準偏差）を計算
    mean_return = ticker_data['Daily Return'].mean()  # 平均リターン
    risk = ticker_data['Daily Return'].std()  # 標準偏差（リスク）
    
    # シャープレシオの計算
    # 日次リターンに対応するリスクフリー金利を引く
    sharpe_ratio = (mean_return - risk_free_rate_per_day) / risk
    # 銘柄ごとのシャープレシオを保存
    ticker_data['sharpe_ratio'] = sharpe_ratio  

    # データフレームをリストに格納
    ticker_data["Stock"] = ticker
    result_list.append(ticker_data)

# 全データを連結
df = pd.concat(result_list)
df = df.set_index(["Stock", "date"]) 

#### シャープレシオの比較チャート（Plotly）

各銘柄のシャープレシオを `plotly.graph_objs.Bar` を用いて棒グラフで比較します。  
棒が高い銘柄ほど同じリスク水準で多くの超過リターンを得ており、投資効率が優れていることを示します。  

棒グラフはスカラー値（期間全体のシャープレシオ）を銘柄ごとに1本表示します。

In [17]:
import plotly.graph_objs as go


# ラベルを等間隔で設定（例: 毎月末の日付を表示）
tick_vals = pd.date_range(
    start=df.index.get_level_values("date").min(),
    end=df.index.get_level_values("date").max(),
    freq="ME"  # 月末
)
tick_vals = [pd.Timestamp(df.index.get_level_values("date").min())] + list(tick_vals) + [pd.Timestamp(df.index.get_level_values("date").max())]
tick_vals = sorted(set(tick_vals))  # 重複を排除し昇順に並べる

# Plotlyを使用して可視化
fig = go.Figure()

for i, ticker in enumerate(codes):
    # 各銘柄のデータを抽出
    ticker_data = df.loc[ticker]
    # 銘柄コードから企業名を取得（なければコードを使用）
    name = company_names[i]['name']
    # グラフに追加
    fig.add_trace(go.Bar(
        x=[name],
        y=ticker_data['sharpe_ratio'],
        name=name
    ))

# グラフの設定
fig.update_layout(
    title="シャープレシオ比較",
    xaxis=dict(
        title="銘柄",
        showline=True,
        showgrid=True,
    ),
    yaxis=dict(
        title="シャープレシオ",
        showline=True
    ),
    template="plotly_white",
)

# グラフを表示
fig.show()
